# Chapter 2: Supervised Learning
## *Introduction to Machine Learning with Python*  Müller & Guido

---
> **Ringkasan Bab:** Bab ini membahas algoritma-algoritma Supervised Learning secara mendalam dari yang sederhana (k-NN, Linear Regression) hingga yang kompleks (Gradient Boosting, SVM). Setiap algoritma dijelaskan cara kerjanya, kapan digunakan, dan kelebihan/kekurangannya.

## 2.1 Konsep Dasar Supervised Learning

**Supervised Learning** = belajar dari data berlabel (pasangan input-output).

**Dua tipe utama:**

| Tipe | Output | Contoh |
|---|---|---|
| **Klasifikasi** | Kategori/kelas diskrit | Spam/not spam, jenis bunga |
| **Regresi** | Nilai kontinu | Harga rumah, suhu besok |

**Terminologi penting:**
- **Fitur (Features/X):** variabel input yang digunakan untuk prediksi
- **Target (Label/y):** variabel output yang ingin diprediksi
- **Generalization:** kemampuan model bekerja baik pada data **baru** yang belum pernah dilihat
- **Overfitting:** model terlalu hafal data training, performa buruk di data baru
- **Underfitting:** model terlalu sederhana, bahkan buruk di data training

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_boston, make_classification
from sklearn.model_selection import train_test_split

# Dataset klasifikasi: Breast Cancer Wisconsin
cancer = load_breast_cancer()
print("=== Breast Cancer Dataset ===")
print("Shape data   :", cancer.data.shape)
print("Nama kelas   :", cancer.target_names)
print("Jumlah fitur :", len(cancer.feature_names))

## 2.2 k-Nearest Neighbors (k-NN)

### Cara Kerja:
- Simpan semua data training
- Untuk prediksi: cari **k titik training terdekat** dari data baru
- **Klasifikasi:** ambil kelas mayoritas dari k tetangga
- **Regresi:** ambil rata-rata nilai dari k tetangga

### Parameter Penting:
- `n_neighbors` (k): semakin besar k → model makin sederhana (lebih smooth)
- `metric`: cara mengukur jarak (default: Euclidean)

### Trade-off k:
- **k kecil (1):** model kompleks → risiko overfitting
- **k besar:** model sederhana → risiko underfitting

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=66
)

# Bandingkan akurasi untuk berbagai nilai k
train_scores = []
test_scores  = []
k_values = range(1, 11)

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    test_scores.append(knn.score(X_test, y_test))

# Plot
plt.figure(figsize=(9, 5))
plt.plot(k_values, train_scores, 'o-', label='Training accuracy', color='blue')
plt.plot(k_values, test_scores,  's-', label='Test accuracy',     color='red')
plt.xlabel('Nilai k (n_neighbors)')
plt.ylabel('Akurasi')
plt.title('k-NN: Pengaruh Nilai k terhadap Akurasi')
plt.legend()
plt.xticks(k_values)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_k = k_values[np.argmax(test_scores)]
print(f"\nNilai k terbaik : {best_k}")
print(f"Akurasi test    : {max(test_scores):.2%}")

## 2.3 Linear Models

Model linear membuat prediksi menggunakan **kombinasi linear** dari fitur input:

$$\hat{y} = w_0 + w_1 x_1 + w_2 x_2 + \ldots + w_p x_p$$

- $w_0$ = intercept (bias)
- $w_1, \ldots, w_p$ = koefisien/bobot setiap fitur
- Model belajar nilai $w$ yang optimal dari data training

### 2.3.1 Linear Regression (untuk Regresi)

- Meminimalkan **Mean Squared Error (MSE)** antara prediksi dan nilai aktual
- Tidak ada regularisasi → bisa overfit pada dataset kecil dengan banyak fitur

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.datasets import make_regression

# Buat dataset regresi sintetis
X, y = make_regression(n_samples=100, n_features=1, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

lr = LinearRegression()
lr.fit(X_train, y_train)

print("Koefisien (slope)  :", lr.coef_)
print("Intercept          :", lr.intercept_)
print(f"R² Training        : {lr.score(X_train, y_train):.4f}")
print(f"R² Test            : {lr.score(X_test,  y_test):.4f}")

# Visualisasi
plt.figure(figsize=(8, 5))
plt.scatter(X_train, y_train, color='blue', alpha=0.5, label='Training data')
plt.scatter(X_test,  y_test,  color='red',  alpha=0.5, label='Test data')
x_line = np.linspace(X.min(), X.max(), 100).reshape(-1,1)
plt.plot(x_line, lr.predict(x_line), color='black', linewidth=2, label='Garis regresi')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Linear Regression')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.3.2 Ridge Regression (L2 Regularization)

- Menambahkan **penalti L2** pada koefisien: $\text{Loss} = \text{MSE} + \alpha \sum w_i^2$
- Parameter `alpha`: semakin besar → koefisien makin kecil → model lebih sederhana
- **Semua fitur tetap digunakan**, hanya koefisiennya yang diperkecil

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.datasets import load_boston

# Boston Housing Dataset
boston = load_boston()
X_train, X_test, y_train, y_test = train_test_split(
    boston.data, boston.target, random_state=0
)

for alpha in [0.1, 1.0, 10.0, 100.0]:
    ridge = Ridge(alpha=alpha).fit(X_train, y_train)
    print(f"Alpha={alpha:6.1f} | Train R²: {ridge.score(X_train,y_train):.3f} | Test R²: {ridge.score(X_test,y_test):.3f}")

### 2.3.3 Lasso Regression (L1 Regularization)

- Menambahkan **penalti L1**: $\text{Loss} = \text{MSE} + \alpha \sum |w_i|$
- **Fitur tidak relevan dapat di-set ke nol** → otomatis feature selection
- Berguna saat ada banyak fitur dan hanya sedikit yang benar-benar informatif

In [ ]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=10000).fit(X_train, y_train)
print(f"Lasso (alpha=0.01)")
print(f"Train R²         : {lasso.score(X_train, y_train):.3f}")
print(f"Test R²          : {lasso.score(X_test, y_test):.3f}")
print(f"Fitur digunakan  : {np.sum(lasso.coef_ != 0)} dari {boston.data.shape[1]}") 

### 2.3.4 Logistic Regression (untuk Klasifikasi)

Meskipun namanya "Regression", **Logistic Regression** adalah algoritma **klasifikasi**.
- Menggunakan fungsi sigmoid untuk menghasilkan probabilitas kelas
- Parameter `C`: kebalikan dari kekuatan regularisasi (C besar = regularisasi lemah)

In [ ]:
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42
)

for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    logreg = LogisticRegression(C=C, max_iter=10000).fit(X_train, y_train)
    print(f"C={C:6.3f} | Train: {logreg.score(X_train,y_train):.3f} | Test: {logreg.score(X_test,y_test):.3f}")

## 2.4 Decision Trees

Decision Tree membagi data secara **rekursif** berdasarkan pertanyaan ya/tidak pada fitur.

**Cara kerja:**
1. Pilih fitur & threshold yang paling baik memisahkan kelas
2. Bagi data menjadi dua cabang
3. Ulangi sampai mencapai kondisi berhenti (kedalaman max, jumlah sampel min, dll)

**Parameter penting:**
- `max_depth`: kedalaman maksimum pohon
- `min_samples_leaf`: jumlah minimum sampel di daun

> ⚠️ Decision Tree tanpa pembatasan cenderung **overfit** karena bisa hafal seluruh data training.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42
)

print("max_depth | Train Acc | Test Acc")
print("-" * 35)
for depth in [None, 2, 3, 4, 5]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(X_train, y_train)
    label = str(depth) if depth else 'None(full)'
    print(f"{label:9} | {tree.score(X_train,y_train):.3f}     | {tree.score(X_test,y_test):.3f}")

In [ ]:
# Feature importance dari Decision Tree
tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)

# Top 10 fitur terpenting
importances = tree.feature_importances_
top_idx = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(10, 5))
plt.bar(range(10), importances[top_idx], color='steelblue')
plt.xticks(range(10), cancer.feature_names[top_idx], rotation=45, ha='right')
plt.title('Feature Importance - Decision Tree (max_depth=4)')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 2.5 Random Forests

**Random Forest** = kumpulan banyak Decision Tree yang bekerja bersama (**ensemble method**).

**Cara kerja:**
1. Buat banyak Decision Tree yang berbeda-beda
2. Setiap tree dilatih pada **subset data** yang dipilih acak (bootstrapping)
3. Setiap tree hanya boleh memilih dari **subset fitur** yang acak di setiap split
4. Prediksi akhir = voting mayoritas (klasifikasi) atau rata-rata (regresi)

**Keunggulan:**
- Lebih robust dari single decision tree
- Tidak mudah overfit
- Sangat baik untuk banyak kasus dunia nyata

**Parameter penting:**
- `n_estimators`: jumlah pohon (lebih banyak = lebih baik, tapi lebih lambat)
- `max_features`: jumlah fitur yang dipertimbangkan di setiap split
- `max_depth`: kedalaman tiap pohon

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=0)
rf.fit(X_train, y_train)

print(f"Akurasi Training : {rf.score(X_train, y_train):.3f}")
print(f"Akurasi Test     : {rf.score(X_test, y_test):.3f}")

# Feature importance
importances_rf = rf.feature_importances_
top_idx_rf = np.argsort(importances_rf)[::-1][:10]

plt.figure(figsize=(10, 5))
plt.bar(range(10), importances_rf[top_idx_rf], color='forestgreen')
plt.xticks(range(10), cancer.feature_names[top_idx_rf], rotation=45, ha='right')
plt.title('Feature Importance - Random Forest (100 trees)')
plt.ylabel('Importance')
plt.tight_layout()
plt.show()

## 2.6 Gradient Boosting

**Gradient Boosting** membangun pohon secara **sekuensial**  setiap pohon baru belajar memperbaiki kesalahan pohon sebelumnya.

**Perbedaan dengan Random Forest:**

| | Random Forest | Gradient Boosting |
|---|---|---|
| **Cara bangun** | Paralel (independen) | Sekuensial (saling bergantung) |
| **Kedalaman pohon** | Dalam | Dangkal (3-5 level) |
| **Kecepatan training** | Lebih cepat | Lebih lambat |
| **Akurasi** | Baik | Sering lebih baik |
| **Sensitif overfitting** | Kurang | Lebih sensitif |

**Parameter penting:**
- `learning_rate`: ukuran langkah koreksi (lebih kecil = lebih hati-hati)
- `n_estimators`: jumlah pohon
- `max_depth`: kedalaman tiap pohon (biasanya 3-5)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gbm = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
gbm.fit(X_train, y_train)

print(f"Gradient Boosting")
print(f"Akurasi Training : {gbm.score(X_train, y_train):.3f}")
print(f"Akurasi Test     : {gbm.score(X_test, y_test):.3f}")

## 2.7 Support Vector Machines (SVM)

**SVM** mencari **hyperplane** (garis/bidang pemisah) yang memaksimalkan **margin** antara dua kelas.

**Konsep penting:**
- **Support vectors:** titik data yang paling dekat dengan hyperplane
- **Margin:** jarak antara hyperplane dan support vectors terdekat
- **Kernel trick:** memproyeksikan data ke dimensi lebih tinggi agar bisa dipisahkan secara linear

**Parameter penting:**
- `C`: trade-off antara margin dan kesalahan klasifikasi
- `kernel`: jenis kernel (`linear`, `rbf`, `poly`)
- `gamma`: seberapa jauh pengaruh satu titik data (untuk kernel RBF)

> ⚠️ **SVM sangat sensitif terhadap skala fitur!** Selalu lakukan normalisasi sebelum menggunakan SVM.

In [ ]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

# Normalisasi data terlebih dahulu
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# SVM dengan kernel RBF
svm = SVC(kernel='rbf', C=1.0, gamma='scale')
svm.fit(X_train_scaled, y_train)

print(f"SVM (RBF Kernel, C=1.0)")
print(f"Akurasi Training : {svm.score(X_train_scaled, y_train):.3f}")
print(f"Akurasi Test     : {svm.score(X_test_scaled,  y_test):.3f}")

## 2.8 Neural Networks (Deep Learning Dasar)

**Multi-Layer Perceptron (MLP)** adalah jaringan saraf tiruan yang terdiri dari:
- **Input layer:** menerima fitur
- **Hidden layer(s):** transformasi non-linear
- **Output layer:** menghasilkan prediksi

**Parameter penting:**
- `hidden_layer_sizes`: jumlah dan ukuran hidden layer, misal `(100,)` atau `(100, 50)`
- `activation`: fungsi aktivasi (`relu`, `tanh`)
- `alpha`: regularisasi L2

In [ ]:
from sklearn.neural_network import MLPClassifier

mlp = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=42)
mlp.fit(X_train_scaled, y_train)

print(f"MLP Neural Network (1 hidden layer, 100 neurons)")
print(f"Akurasi Training : {mlp.score(X_train_scaled, y_train):.3f}")
print(f"Akurasi Test     : {mlp.score(X_test_scaled,  y_test):.3f}")

## 2.9 Perbandingan Semua Algoritma

| Algoritma | Train Acc | Test Acc | Kelebihan | Kekurangan |
|---|---|---|---|---|
| k-NN | - | - | Sederhana, intuitif | Lambat saat prediksi, sensitif skala |
| Linear Regression | - | - | Cepat, interpretatif | Hanya hubungan linear |
| Ridge/Lasso | - | - | Menangani overfitting | Masih linear |
| Logistic Reg | - | - | Cepat, probabilistik | Hanya linear |
| Decision Tree | - | - | Interpretatif | Mudah overfit |
| Random Forest | - | - | Robust, akurat | Kurang interpretatif |
| Gradient Boosting | - | - | Akurasi tinggi | Lambat, banyak parameter |
| SVM | - | - | Efektif di high-dim | Sensitif skala, lambat |
| MLP | - | - | Fleksibel | Banyak parameter, butuh banyak data |

##  Ringkasan Bab 2

| Konsep | Penjelasan |
|---|---|
| **Overfitting** | Model terlalu kompleks → bagus di training, buruk di test |
| **Underfitting** | Model terlalu sederhana → buruk di keduanya |
| **Regularisasi** | Teknik untuk mencegah overfitting (Ridge=L2, Lasso=L1) |
| **Feature Importance** | Seberapa informatif tiap fitur untuk prediksi |
| **Scaling** | SVM & Neural Network butuh fitur yang dinormalisasi |
| **Ensemble** | Gabungan banyak model (Random Forest, Gradient Boosting) |

---
 **Next:** Chapter 3 — Unsupervised Learning